In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 05 · Hosted API or your own GPUs — the Mistral provider and a vLLM fleet

**What you'll learn.** Notebooks 01–04 worked the anchor scenario on a hosted API, where the only way to pay for
tokens is per token. An open-weight model adds a second way: run it yourself on a vLLM fleet, sized for the peak and
paid for by the hour whether busy or not. This notebook takes the same 100,000 conversations a day to Mistral's API
and to a fleet of Ministral 14B replicas: what one replica delivers, how many you need, the GPU price at which the
fleet beats the API, and what overload looks like on a fleet — no 429s, just everyone slower. Code:
`scalelab/mistral.py` (the demand and the money), `scalelab/serving.py` (one replica), the `ServerPool` and
`HybridBackend` backends in `scalelab/model.py`, and `scalelab/sim.py`. The primer is
`docs/mistral/01-scaling-primer.md`.

> **In a design review.** *"The unit of work is a turn; 100k conversations a day is about 14 million tokens a minute
> at peak. On the API that is a 60-RPS / 19 M-TPM limit I have to ask for; on our own GPUs it is eleven H100s sized
> for the peak — and at this volume the two cost about the same on committed GPUs, so the data-residency requirement
> decides, and the arithmetic prices that choice."*

Every number here is simulated or estimated from first principles (labelled so); prices are list prices dated
2026-09-19 — verify before quoting.

In [ ]:
import os, sys, asyncio, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The same demand on Mistral's API

The chain is notebook 01's: conversations → turns → calls → tokens per minute; Little's law for what is in flight.
What changes is the price per token and the limit. Mistral's paid-tier limits are shown only in the console, so the
capacity plan **is** the rate-limit request you send.

In [ ]:
from scalelab.mistral import Scenario, plan, cost_per_call, fleet, FREE_TIER
from scalelab.serving import replica, GPU_USD_PER_HOUR
s = Scenario()          # the anchor scenario: 100k conversations/day, Mistral Small 4 with 10 % on Medium 3.5
p = plan(s)
tokens = pd.DataFrame(p["tokens"]).T
print("peak total TPM (M):", round(tokens.total_tpm["peak"] / 1e6, 1), "| vs the assumed limit:", round(tokens.vs_limit["peak"], 2), "x")
print("the request to send:", p["hosted"]["limit_request"])
pd.DataFrame({"$/conversation": p["hosted"]["cost_per_conversation"], "$/month": p["hosted"]["monthly_usd"]}).round(4)

Cached input is billed at 10 %; the Priority Tier is +75 %, a regional endpoint +10 % (2026-09-19, verify). Route
and cache first: every later percentage applies to the new baseline.

## 2. What one replica delivers

A vLLM replica has two budgets. **Memory**: weights plus KV cache, so the KV bytes per token decide how many calls
can be resident (grouped-query attention in Ministral 14B: 160 KiB per token; latent attention in Small 4: 22.5 KiB).
**Time per decode step**: every step streams the weights it touches plus the KV of every running sequence through
HBM, so the time per output token (TPOT) grows with the batch while aggregate tokens/s grows until the step is
compute-bound. *The latency you want is the batch you can afford.* These are first-principles estimates (layer 04's
`minengine.perf` is the reference step-time model) — replace them with `vllm bench serve` on the GPU you are given.

In [ ]:
pd.DataFrame([replica(m, g).describe() for m, g in [("ministral-14b", "h100"), ("ministral-14b", "l40s"),
                                                      ("mistral-small-4", "h100"), ("mistral-small-4", "h200"), ("mistral-large-3", "h200")]]).set_index("model")

In [ ]:
ctx = s.input_tokens + s.output_tokens
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for key, gpu in [("ministral-14b", "h100"), ("mistral-small-4", "h100")]:
    r = replica(key, gpu); batches = range(1, 129)
    axes[0].plot(batches, [r.tpot(b, ctx) * 1e3 for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
    axes[1].plot(batches, [r.tokens_per_s(b, ctx) for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
axes[0].axhline(s.target_tpot_s * 1e3, ls="--", c="grey"); axes[0].set_ylabel("TPOT (ms, estimated)"); axes[0].set_xlabel("sequences in the batch")
axes[1].set_ylabel("aggregate tokens / s (estimated)"); axes[1].set_xlabel("sequences in the batch")
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 3. The fleet, and where it beats the API

Size for the peak at the target TPOT, add headroom (and never fewer than two replicas), price the GPUs by the hour.
Both bills grow with volume — the API's linearly, the fleet's in steps — so the break-even is not a volume but a
**GPU price**: the hourly rate at which the peak-sized fleet costs what the API would charge for the same
conversations. Volume matters only at the bottom, where the two-replica floor is not yet amortised.

In [ ]:
f = p["self_hosted"]
print(f"{f['model']} on {f['gpu']}: batch {f['batch']} → TPOT {f['tpot_ms']:.0f} ms, {f['tok_s_per_replica']:,.0f} tok/s per replica; "
      f"a call takes {f['call_seconds']:.1f} s, a turn {f['turn_seconds']:.1f} s (hosted: {s.turn_seconds:g} s)")
print("replicas (avg / peak / incident):", f["replicas"], "→ in-flight cap for the peak fleet:", round(f["max_inflight_turns_peak_fleet"]))
print(f"break-even GPU price vs the hosted planning mix: ${f['breakeven_gpu_usd_per_hour']:.2f} per GPU-hour")
pd.DataFrame({"$/GPU-h": GPU_USD_PER_HOUR[s.gpu], "peak fleet $/month": f["monthly_usd_peak_by_price"], "$/conversation": f["cost_per_conversation_by_price"],
              "vs hosted": f["vs_hosted_by_price"], "floor conversations/day": f["floor_conversations_per_day_by_price"]}).round(4)

In [ ]:
hosted_per_conv = p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
volumes = np.arange(5_000, 300_001, 5_000)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(volumes, volumes * hosted_per_conv * 30.4 / 1e3, label="Mistral API (planning mix, cached)", lw=2)
for price in ("aws on-demand", "gcp 3-year", "neocloud"):
    fleets = [fleet(Scenario(conversations_per_day=int(v), gpu_price=price))["monthly_usd"]["peak"] / 1e3 for v in volumes]
    ax.step(volumes, fleets, where="post", label=f"fleet sized for peak, {price} (${GPU_USD_PER_HOUR[s.gpu][price]:.2f}/GPU-h)")
ax.set_xlabel("conversations per day"); ax.set_ylabel("k$ / month"); ax.set_title("The API is a line; a fleet is a staircase whose slope is the GPU price"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 4. On a fleet nobody is refused — everybody is slower

One Ministral 14B replica on an H100 (simulated). Send *k* calls at once: no 429, no error — the batch grows, the
time per token grows with it, and every call in the batch pays. Past the batch where the target TPOT holds (dashed
line), per-call latency rises faster than throughput.

In [ ]:
from scalelab.model import ServerPool
rows = []
for k in (1, 8, 16, 24, 32, 48, 64, 96):
    CLOCK.reset(0.02)
    srv = ServerPool(replica("ministral-14b", "h100"), replicas=1, context_tokens=5_200, shared_prefix_tokens=3_000)
    rng = random.Random(0); t0 = CLOCK.now()
    lat = await asyncio.gather(*(srv.serve(5_000, 3_000, 300, rng) for _ in range(k)))
    rows.append({"concurrent calls": k, "p50 call latency (s)": np.median([l for l, _ in lat]), "tokens/s served": k * 300 / (CLOCK.now() - t0), "errors": srv.rejected})
df = pd.DataFrame(rows).set_index("concurrent calls").round(2); print(df)
ax = df[["p50 call latency (s)"]].plot(figsize=(6.5, 3.4), marker="o", title="One replica: latency vs concurrency (simulated)")
ax.axvline(srv.target_batch, ls="--", c="grey"); ax.set_ylabel("seconds"); ax.grid(alpha=.3); plt.tight_layout()

The admission controller therefore reads one more signal on a fleet: **saturation**, running + waiting against the
batch the latency target allows (level 1 at 0.8, level 2 at 1.0 — the Kubernetes inference gateway's saturation
detector uses the same idea: queue depth ≥ 5 or KV usage ≥ 0.8, verify). On the API that signal is the pool's use.

## 5. The load test, on each backend

The same users against the API, the fleet, and the hybrid (the fleet first, the API when the fleet is saturated).
`make_setup(mode)` picks the backend; called without one it reads `SCALELAB_BACKEND` (`hosted`, `local` or `hybrid`)
and `SCALELAB_PROVIDER` (`gemini` or `mistral`), so `SCALELAB_BACKEND=local jupyter lab` runs notebook 04's load test
against a fleet without editing it (its exercise (d) then checks the fleet's target, as (f) below does).

In [ ]:
from scalelab.sim import make_setup, simulate, compare
results = {}
for name, mode, kw, users in [("API, moderate (20 users)", "hosted", dict(provider="mistral", pool_tpm=3_000_000), 20),
                              ("API, overload, naive", "hosted", dict(provider="mistral", pool_tpm=3_000_000, naive=True), 120),
                              ("API, overload, degrade + cap", "hosted", dict(provider="mistral", pool_tpm=3_000_000), 120),
                              ("fleet, moderate (20 users)", "local", dict(replicas=2), 20),
                              ("fleet, overload, naive", "local", dict(replicas=2, naive=True), 120),
                              ("fleet, overload, degrade + cap", "local", dict(replicas=2), 120),
                              ("hybrid, overload", "hybrid", dict(replicas=2, pool_tpm=3_000_000), 120)]:
    CLOCK.reset(0.02)
    results[name] = await simulate(make_setup(mode, **kw), users=users, duration_s=90, think_s=5)
summary = compare(results).round(4)
summary[["turns", "completed", "shed_rate", "p50_s", "p95_s", "pushback_calls", "failed", "degraded_share", "max_batch", "api_share_of_calls", "cost_per_turn_usd"]]

In [ ]:
latency_cdfs({k: v for k, v in results.items() if not k.startswith("API")}, "Turn latency — the fleet and the hybrid (simulated)")

In [ ]:
timeline(results["fleet, overload, naive"], "fleet, naive: no errors, no 429s — the batch climbs and everyone slows down")

In [ ]:
timeline(results["hybrid, overload"], "hybrid: the batch holds near the target; the excess goes to the API")

On the API the naive run is notebook 04's feedback loop (429s → retries → longer turns → more in flight). On the
fleet it takes its second form: zero errors, zero pushback, a batch far past the target and a p95 well above the
moderate run — a green dashboard while every user waits. The cap and the saturation signal bound the batch; the
hybrid keeps the batch at the target and pays the API for the rest, and `cost_per_turn_usd` is what that costs.

## 6. From measurements to vLLM and Kubernetes settings

Take the per-replica throughput at the target batch and the turn duration from the moderate runs, apply Little's law
at the planned peak, add headroom, divide by what one pod or one replica can hold.

In [ ]:
fleet_turn = results["fleet, moderate (20 users)"].summary()["p95_s"]
print(f"measured p95 turn on the fleet: {fleet_turn:.1f} s (simulated; the plan assumed {f['turn_seconds']:.1f} s)")
for level in ("average", "peak", "incident"):
    turns_s, calls_s = p["rates"][level]["turns_per_s"], p["rates"][level]["calls_per_s"]
    pods = np.ceil(turns_s * fleet_turn * 1.4 / 80)
    by_tokens = calls_s * s.output_tokens / f["tok_s_per_replica"]          # decode throughput
    by_seqs = calls_s * f["call_seconds"] / f["batch"]                       # Little: calls in flight ÷ batch per replica
    print(f"{level:9s} {turns_s:5.1f} turns/s → {pods:2.0f} orchestrator pods (80 turns each, ×1.4); "
          f"{calls_s:5.1f} calls/s → {by_tokens:4.1f} replicas by tokens/s, {by_seqs:4.1f} by batch → {np.ceil(max(by_tokens, by_seqs) * 1.3):2.0f} vLLM replicas (×1.3)")

| Setting | Value (Ministral 14B on H100) | Why |
|---|---|---|
| vLLM replicas | 11 at peak (static), KEDA to 14 for headroom | tokens/s per replica at the target batch; a GPU replica's cold start is minutes, so the peak is provisioned, not autoscaled |
| `--max-num-seqs` | 48 (2× the target batch of 24) | room above the target for bursts; beyond it TPOT is being traded away |
| `--max-num-queued-reqs` | ≈ 12 per replica | turns a queue into a 503 the gateway can act on; without it vLLM queues forever |
| `--max-model-len` | 16,384 | frees KV memory versus the model's long default; the turn never needs more |
| `--kv-cache-dtype fp8` | on | halves KV bytes per token → about twice the resident sequences |
| in-flight cap (gateway) | about 290 turns for the peak fleet; 175 on a 20 M TPM API limit | Little's law from the fleet's or the limit's capacity; tune from the load test |
| KEDA (fleet) | `vllm:num_requests_waiting` > 5 or `vllm:kv_cache_usage_perc` > 0.8 | the inference gateway's saturation thresholds |

Flags and thresholds as of vLLM 0.29–0.30 and the Gateway API Inference Extension, September 2026 (verify). The
Kubernetes reference architecture these settings belong to is the last section of `docs/02-reference-architecture.md`.

## 7. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn

Fill in each function (replace the `todo()` call), then run the check cell below it. The practice notebook runs end
to end with the exercises untouched; the solutions notebook has every check passing. The checks compare your
function with the lab's model over many inputs, not one.

#### (a) KV cache and resident sequences

Bytes per token for grouped-query attention: 2 (K and V) × layers × kv_heads × head_dim × bytes per element. The
number of sequences a KV budget holds: the shared prefix is stored once, every sequence pays for its own tokens,
capped at `max_num_seqs`.

In [ ]:
def kv_bytes(layers, kv_heads, head_dim, bytes_per_elem=2):
    return todo()

def max_sequences(kv_budget_gb, kv_bytes_per_token, context_tokens, prefix_tokens=0, max_num_seqs=128):
    return todo()

In [ ]:
def _a():
    from scalelab.serving import kv_bytes_per_token
    for layers, heads, dim, b in [(40, 8, 128, 2), (34, 8, 128, 2), (61, 4, 192, 1), (12, 2, 64, 4)]:
        assert kv_bytes(layers, heads, dim, b) == kv_bytes_per_token(layers, heads, dim, b), (layers, heads, dim, b)
    for m, g in [("ministral-14b", "h100"), ("ministral-8b", "l40s"), ("mistral-small-4", "h200"), ("mistral-large-3", "b200")]:
        for fp8 in (False, True):
            r = replica(m, g, kv_dtype_bytes=1 if fp8 else 2)
            for ctx, prefix in [(5_200, 0), (5_200, 2_700), (32_000, 3_000), (1_000, 0)]:
                got = max_sequences(r.kv_budget_gb, r.model.kv_bytes_per_token * r.kv_dtype_bytes / 2, ctx, prefix)
                assert got == r.max_seqs(ctx, prefix), f"{m} on {g}, fp8 KV={fp8}, {ctx} tokens with a {prefix} prefix: {got} vs {r.max_seqs(ctx, prefix)}"
    r = replica("ministral-14b", "h100")
    print(f"   Ministral 14B on one H100: {r.max_seqs(5_200)} calls of 5.2k tokens resident without prefix caching, "
          f"{r.max_seqs(5_200, 2_700)} with (the max_num_seqs cap), {replica('ministral-14b', 'h100', kv_dtype_bytes=1).max_seqs(5_200)} with FP8 KV")
check("a: KV cache", _a)

#### (b) The break-even GPU price

The peak-sized fleet burns `gpus × 730` GPU-hours a month to serve `conversations_per_day × 30.4` conversations. At
what hourly GPU price does that equal the API's price per conversation? And below what daily volume is a minimum
fleet of `min_gpus` GPUs not amortised at a given price?

In [ ]:
def breakeven_gpu_usd_per_hour(gpus, conversations_per_day, hosted_usd_per_conversation):
    return todo()

def floor_conversations_per_day(min_gpus, usd_per_gpu_hour, hosted_usd_per_conversation):
    return todo()

In [ ]:
def _b():
    for kw in [{}, dict(conversations_per_day=30_000), dict(conversations_per_day=250_000, premium_share=0.3),
               dict(open_model="mistral-small-4", gpu="h200", gpu_price="neocloud"), dict(output_tokens=600, turns_per_conversation=9)]:
        sc = Scenario(**kw); pl = plan(sc); fl = pl["self_hosted"]
        mix = pl["hosted"]["cost_per_conversation"][pl["hosted"]["planning_mix"]]
        be = breakeven_gpu_usd_per_hour(fl["gpus"]["peak"], sc.conversations_per_day, mix)
        assert abs(be - fl["breakeven_gpu_usd_per_hour"]) < 1e-6 * be, (kw, be, fl["breakeven_gpu_usd_per_hour"])
        tp = fl["gpus"]["peak"] // fl["replicas"]["peak"]
        for price_key, usd in GPU_USD_PER_HOUR[sc.gpu].items():
            want = fl["floor_conversations_per_day_by_price"][price_key]
            assert abs(floor_conversations_per_day(sc.min_replicas * tp, usd, mix) - want) < 1e-6 * want, (kw, price_key)
    f, mix = p["self_hosted"], p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
    be = breakeven_gpu_usd_per_hour(f["gpus"]["peak"], s.conversations_per_day, mix)
    print(f"   the fleet beats the API below ${be:.2f}/GPU-hour — at AWS on-demand H100 prices ($6.88) it loses, at 3-year commitments "
          f"($4.86–5.40) it is level, at neocloud prices ($3.75) it wins; at $3.75 two replicas are amortised from "
          f"{floor_conversations_per_day(2, 3.75, mix):,.0f} conversations/day (list prices 2026-09-19, verify)")
check("b: break-even price", _b)

#### (c) Replicas needed

Two constraints, take the larger: decode throughput (calls/s × output tokens ÷ tokens/s per replica) and resident
calls (calls/s × call seconds ÷ batch per replica, Little's law); then × headroom, round up, never below `minimum`.

In [ ]:
def replicas_needed(calls_per_s, output_tokens, tok_s_per_replica, call_seconds, batch, headroom=1.3, minimum=2):
    return todo()

In [ ]:
def _c():
    for kw in [{}, dict(conversations_per_day=5_000), dict(conversations_per_day=400_000), dict(open_model="mistral-small-4", gpu="h200"),
               dict(target_tpot_s=0.035, headroom=1.5, min_replicas=3), dict(gpu="l40s", target_tpot_s=0.04, output_tokens=400)]:
        sc = Scenario(**kw); fl = fleet(sc); pl = plan(sc)
        for level in ("average", "peak", "incident"):
            got = replicas_needed(pl["rates"][level]["calls_per_s"], sc.output_tokens, fl["tok_s_per_replica"], fl["call_seconds"], fl["batch"],
                                  sc.headroom, sc.min_replicas)
            assert got == fl["replicas"][level], f"{kw or 'anchor'} at {level}: {got} replicas, the model sizes {fl['replicas'][level]}"
    print("   anchor scenario:", p["self_hosted"]["replicas"])
check("c: replicas", _c)

#### (d) The in-flight cap from the fleet

How many turns can be in flight before the fleet passes its target batch: replicas × batch × turn seconds ÷ (calls
per turn × call seconds).

In [ ]:
def cap_from_fleet(replicas, batch, turn_seconds, calls_per_turn, call_seconds):
    return todo()

In [ ]:
def _d():
    for kw in [{}, dict(conversations_per_day=40_000), dict(calls_per_turn=3.1, tool_seconds_per_turn=2.5), dict(gpu="h200", target_tpot_s=0.03)]:
        sc = Scenario(**kw); fl = fleet(sc)
        got = cap_from_fleet(fl["replicas"]["peak"], fl["batch"], fl["turn_seconds"], sc.calls_per_turn, fl["call_seconds"])
        assert abs(got - fl["max_inflight_turns_peak_fleet"]) < 1e-6 * got, (kw or "anchor", got, fl["max_inflight_turns_peak_fleet"])
    print(f"   the peak fleet holds about {p['self_hosted']['max_inflight_turns_peak_fleet']:.0f} turns in flight before it passes its target batch")
check("d: cap from the fleet", _d)

#### (e) The spill-over decision

Route a model call: `"fleet"` while the fleet's saturation is below `spill_at`; otherwise `"api"` if the call carries
no personal data (the residency policy) and the API's breaker is closed; otherwise `"degrade"` (stay on the fleet with
shorter answers). Priority callers (≤ 2) always get the fleet.

In [ ]:
def route(saturation, has_pii, api_breaker_closed, priority=5, spill_at=0.9):
    return todo()

In [ ]:
def _e():
    assert route(0.5, True, True) == "fleet"
    assert route(0.95, False, True) == "api"
    assert route(0.95, True, True) == "degrade"           # personal data never leaves your environment
    assert route(0.95, False, False) == "degrade"         # the API is broken: stay home
    assert route(1.5, True, True, priority=1) == "fleet"  # priority traffic gets the GPUs
    assert route(0.7, False, True, spill_at=0.6) == "api" and route(0.9, False, True) == "api"   # the threshold is inclusive
    for sat in np.linspace(0, 2, 41):                     # below the threshold nothing else matters
        if sat < 0.9:
            assert {route(sat, pii, ok) for pii in (True, False) for ok in (True, False)} == {"fleet"}, sat
check("e: spill-over", _e)

#### (f) Choose the cap

Pick an in-flight cap for two replicas with 120 users so that p95 ≤ 7.5 s (simulated) and no turn fails; keep
shedding under 45 %. The check runs a simulation with your value.

In [ ]:
MY_CAP = None   # an integer

In [ ]:
async def _f():
    if MY_CAP is None:
        todo()
    CLOCK.reset(0.02)
    r = await simulate(make_setup("local", replicas=2, max_inflight=MY_CAP), users=120, duration_s=60, think_s=5)
    s_ = r.summary()
    print(f"   cap {MY_CAP}: p95 {s_['p95_s']:.1f} s, shed {s_['shed_rate']:.0%}, failed {s_['failed']}, max batch {s_['max_batch']}")
    assert s_["p95_s"] <= 7.5 and s_["failed"] == 0 and s_["shed_rate"] < 0.45
await acheck("f: choose the cap", _f)

## Takeaways for the design review

- Hosted or self-hosted is a price question with a residency answer: quote the break-even GPU price (≈ $5 per
  GPU-hour here: on-demand loses, committed is level, neoclouds win), then say which requirement decides.
- One replica = a KV budget and a bandwidth budget; the batch you run *is* your latency. Size the fleet for the peak
  at the target TPOT, then price it by the hour.
- The naive fleet does not fail — it slows down for everyone with a green dashboard. Cap the queue, watch the batch,
  feed saturation to admission control, and let the API take the spill-over where the data policy allows.

## Verify before the design review

Mistral list prices and model ids (2026-09-19 in `scalelab/mistral.py`), the paid-tier limits in the console, the
replica estimates against `vllm bench serve` on the GPUs you are given, GPU hourly prices in your region, vLLM flags
and defaults (`max_num_seqs` 128, `--max-num-queued-reqs`), and the inference gateway's saturation thresholds.